In [5]:
from pathlib import Path

TRANSCRIPTS_DIR = Path("../transcripts")
SEGMENTS_DIR = Path("../segments")

N_LEVELS = [1, 2, 4, 8, 16, 32, 64]

for label in ["HC", "PT"]:
    (SEGMENTS_DIR / label).mkdir(parents=True, exist_ok=True)

print("Output folders ready:", list(SEGMENTS_DIR.glob("*")))

Output folders ready: [PosixPath('../segments/HC'), PosixPath('../segments/PT')]


In [6]:
def chunk_words(words, n):
    """
    Split a list of words into n contiguous, near-equal chunks.
    Recursively halving into 2, then those into 2 again, etc., down to n
    segments is mathematically equivalent to dividing the whole list into
    n equal contiguous chunks directly -- so we do it directly here.
    """
    total = len(words)
    boundaries = [round(i * total / n) for i in range(n + 1)]
    chunks = []
    for i in range(n):
        start, end = boundaries[i], boundaries[i + 1]
        chunks.append(words[start:end])
    return chunks


def segment_transcript(txt_path: Path, output_dir: Path, n_levels=N_LEVELS):
    """
    Read one transcript, split it at every N level, and write out
    flat files named <stem>_N<N>_seg<k>.txt into output_dir.
    Returns the number of segment files written.
    """
    text = txt_path.read_text(encoding="utf-8")
    words = text.split()   # whitespace split -- simplest, safest tokenization
    stem = txt_path.stem   # e.g. '01_CF56_1'

    n_written = 0
    for n in n_levels:
        if len(words) < n:
            print(f"  !! WARNING: {stem} has only {len(words)} words, cannot split into N={n}. Skipping.")
            continue
        chunks = chunk_words(words, n)
        for i, chunk in enumerate(chunks, 1):
            seg_text = " ".join(chunk)
            out_path = output_dir / f"{stem}_N{n}_seg{i}.txt"
            out_path.write_text(seg_text, encoding="utf-8")
            n_written += 1
    return n_written

In [7]:
## test 
test_file = next((TRANSCRIPTS_DIR / "HC").glob("*.txt"))
print("Testing on:", test_file.name)

out_dir = SEGMENTS_DIR / "HC"
n_written = segment_transcript(test_file, out_dir)
print(f"Wrote {n_written} segment files")

# spot check: N=4 should give 4 roughly equal pieces
for i in range(1, 5):
    p = out_dir / f"{test_file.stem}_N4_seg{i}.txt"
    words = p.read_text(encoding="utf-8").split()
    print(f"  N4_seg{i}: {len(words)} words -> starts: {' '.join(words[:6])}...")

Testing on: 33_CF46_3.txt
Wrote 127 segment files
  N4_seg1: 162 words -> starts: allora dicci un po' cosa fai...
  N4_seg2: 163 words -> starts: in quel momento non riesci a...
  N4_seg3: 163 words -> starts: molto grande e lui la la...
  N4_seg4: 162 words -> starts: meno manager di me stessa e...


In [8]:
# Verify N4 segments reconstruct the original transcript exactly
original_words = test_file.read_text(encoding="utf-8").split()

reconstructed = []
for i in range(1, 5):
    p = out_dir / f"{test_file.stem}_N4_seg{i}.txt"
    reconstructed.extend(p.read_text(encoding="utf-8").split())

print("Original word count:    ", len(original_words))
print("Reconstructed word count:", len(reconstructed))
print("Exact match:", original_words == reconstructed)

Original word count:     650
Reconstructed word count: 650
Exact match: True


In [9]:
all_transcripts = sorted((TRANSCRIPTS_DIR / "HC").glob("*.txt")) + sorted((TRANSCRIPTS_DIR / "PT").glob("*.txt"))
print(f"Found {len(all_transcripts)} transcripts to segment")

total_written = 0
for i, txt_path in enumerate(all_transcripts, 1):
    label = txt_path.parent.name  # 'HC' or 'PT'
    out_dir = SEGMENTS_DIR / label
    n_written = segment_transcript(txt_path, out_dir)
    total_written += n_written
    print(f"[{i}/{len(all_transcripts)}] {label}/{txt_path.name} -> {n_written} files")

print(f"\nDone. {total_written} total segment files written.")

Found 116 transcripts to segment
[1/116] HC/01_CF56_1.txt -> 127 files
[2/116] HC/02_CM57_2.txt -> 127 files
[3/116] HC/03_CF30_3.txt -> 127 files
[4/116] HC/04_CF57_3.txt -> 127 files
[5/116] HC/05_CF41_3.txt -> 127 files
[6/116] HC/06_CF44_2.txt -> 127 files
[7/116] HC/07_CF50_2.txt -> 127 files
[8/116] HC/08_CF42_2.txt -> 127 files
[9/116] HC/09_CF56_3.txt -> 127 files
[10/116] HC/10_CF51_2.txt -> 127 files
[11/116] HC/11_CF44_2.txt -> 127 files
[12/116] HC/12_CF36_1.txt -> 127 files
[13/116] HC/13_CF45_2.txt -> 127 files
[14/116] HC/14_CF49_3.txt -> 127 files
[15/116] HC/15_CF53_3.txt -> 127 files
[16/116] HC/16_CF33_4.txt -> 127 files
[17/116] HC/17_CF55_3.txt -> 127 files
[18/116] HC/18_CM64_3.txt -> 127 files
[19/116] HC/19_CF62_4.txt -> 127 files
[20/116] HC/20_CM51_3.txt -> 127 files
[21/116] HC/21_CF58_3.txt -> 127 files
[22/116] HC/22_CF50_3.txt -> 127 files
[23/116] HC/23_CF55_3.txt -> 127 files
[24/116] HC/24_CM63_3.txt -> 127 files
[25/116] HC/25_CF59_3.txt -> 127 files
[

In [10]:
for label in ["HC", "PT"]:
    n_files = len(list((SEGMENTS_DIR / label).glob("*.txt")))
    print(f"{label}: {n_files} segment files")

HC: 6604 segment files
PT: 8128 segment files
